# 06 Gold DLT Pipeline — Datamart-laag

**Laag:** Gold / Datamart  
**Schema:** `DATAMART`  
**Pipeline-type:** Lakeflow Declarative Pipeline (DLT)

## Wat deze pipeline doet

Leest van `INTEGRATION.order_header` (order-grain) en `INTEGRATION.sales_line`
(regel-grain) en berekent vier Gold Materialised Views in `DATAMART`.

**Bronkeuze:** aggregaten lezen van `order_header` (order-grain) om
`SUM`-over-gedupliceerde-regelrijen te voorkomen.  Detailrijen (meerdere per order) zouden
`order_total`, `order_tax_amount` en `order_discount_amount` vermenigvuldigen als we zouden
aggregeren vanuit `sales_line`.  De brede tabel `sales_lines_wide` leest wél vanuit `sales_line`
— dat is de regel-grain AI/BI-target met afgeleide kolommen, geen aggregaat.

**NULL truck_id / location_id:** beide kolommen zijn `warn`-regels in Silver — rijen
met een ontbrekende waarde passeren het clean-filter en landen in `order_header`.  Ze
verschijnen hier als één `NULL`-rij in het aggregaat (de “Unknown”-bucket).
Dit is opzettelijk.

## Tabellen in deze pipeline

| Tabel | Type | Grain | Beschrijving |
|---|---|---|---|
| `DATAMART.daily_sales_by_truck` | Materialised View | (order_date, truck_id) | KPI: omzet per truck per dag |
| `DATAMART.daily_sales_by_location` | Materialised View | (order_date, location_id) | KPI: omzet per locatie per dag |
| `DATAMART.monthly_revenue_by_currency` | Materialised View | (year_month, order_currency) | Maandtrend per valuta |
| `DATAMART.sales_lines_wide` | Materialised View (Liquid Clustering) | regel | Brede AI/BI-vriendelijke tabel voor Genie & Dashboards |

In [ ]:
%run ./_lib/derivation_helpers

In [ ]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

## Parameters

DLT leest pipeline-parameters uit `spark.conf`.  De `catalog`-parameter wordt doorgegeven
door de DAB-pipelineconfiguratie (`resources/dlt_datamart.yml`).

In [ ]:
catalog = spark.conf.get("pipeline.catalog", "DEMO")
silver_schema = "INTEGRATION"

print(f"Catalog : {catalog}")
print(f"Silver  : {catalog}.{silver_schema}")

## daily_sales_by_truck — Materialised View

Aggregeert `INTEGRATION.order_header` op `(order_date, truck_id)`-grain.

**Kolomspec:**

| Kolom | Type | Berekening |
|---|---|---|
| `order_date` | DATE | `CAST(order_ts AS DATE)` |
| `truck_id` | DECIMAL(38, 0) | groepeersleutel |
| `total_orders` | BIGINT | `COUNT(*)` |
| `total_revenue` | DECIMAL(38, 4) | `SUM(order_total)` |
| `total_tax` | DECIMAL(38, 4) | `SUM(order_tax_amount)` |
| `total_discount` | DECIMAL(38, 4) | `SUM(order_discount_amount)` |
| `avg_order_value` | DECIMAL(38, 4) | `total_revenue / total_orders` |

**Patroon:** `@dp.materialized_view` met `spark.read` (batch-semantiek).
De pipeline herberekent deze volledig bij elke run; Silver-wijzigingen (correcties,
late aankomsten) propageren automatisch bij de volgende verversing.

**NULL truck_id-rijen:** Rijen waarbij `truck_id IS NULL` (orders met onbekende truck die
Silver's warn-regel passeerden) worden opzettelijk bewaard.  Ze verschijnen als één
`truck_id = NULL`-bucket — zichtbare data-attributieproblemen voor analisten.

In [ ]:
@dp.materialized_view(
    name="daily_sales_by_truck",
    comment=(
        "KPI aggregate: daily revenue per truck from INTEGRATION.order_header. "
        "Grain: (order_date, truck_id). NULL truck_id rows kept as Unknown bucket. "
        "Materialised View — fully recomputed on each pipeline run."
    ),
)
def daily_sales_by_truck():
    """Materialised View — dagelijkse omzet-KPI geaggregeerd per truck.

    Leest INTEGRATION.order_header via spark.read (batch-semantiek).  Een
    @dp.materialized_view wordt volledig herberekend bij elke pipeline-run, zodat
    Silver-correcties automatisch propageren.

    Grain: één rij per (order_date, truck_id)-combinatie.  NULL truck_id-waarden
    worden opzettelijk bewaard — ze verschijnen als één Unknown-bucket die
    orders met onbekende truck zichtbaar maakt voor analyst-triage.

    avg_order_value wordt gecast naar DECIMAL(38,4) na de deling om de kolomspec te
    matchen; de deling zelf promoveert naar Double, dus een expliciete cast is vereist.
    """
    oh = spark.read.table(f"{catalog}.{silver_schema}.order_header")

    agg = (
        oh
        .withColumn("order_date", F.col("order_ts").cast("date"))
        .groupBy("order_date", "truck_id")
        .agg(
            F.count("*")                        .cast("bigint")         .alias("total_orders"),
            F.sum("order_total")                .cast("decimal(38,4)") .alias("total_revenue"),
            F.sum("order_tax_amount")           .cast("decimal(38,4)") .alias("total_tax"),
            F.sum("order_discount_amount")      .cast("decimal(38,4)") .alias("total_discount"),
        )
    )

    return agg.withColumn(
        "avg_order_value",
        (F.col("total_revenue") / F.col("total_orders")).cast("decimal(38,4)"),
    ).select(
        "order_date",
        "truck_id",
        "total_orders",
        "total_revenue",
        "total_tax",
        "total_discount",
        "avg_order_value",
    )

## daily_sales_by_location — Materialised View

Aggregeert `INTEGRATION.order_header` op `(order_date, location_id)`-grain.
Dezelfde structuur als `daily_sales_by_truck`, maar gegroepeerd op `location_id`.

**Kolomspec:**

| Kolom | Type | Berekening |
|---|---|---|
| `order_date` | DATE | `CAST(order_ts AS DATE)` |
| `location_id` | DECIMAL(38, 0) | groepeersleutel |
| `total_orders` | BIGINT | `COUNT(*)` |
| `total_revenue` | DECIMAL(38, 4) | `SUM(order_total)` |
| `total_tax` | DECIMAL(38, 4) | `SUM(order_tax_amount)` |
| `total_discount` | DECIMAL(38, 4) | `SUM(order_discount_amount)` |
| `avg_order_value` | DECIMAL(38, 4) | `total_revenue / total_orders` |

**NULL location_id-rijen:** `location_id IS NOT NULL` is een `warn`-regel in Silver —
rijen met een ontbrekende location_id passeren het clean-filter en landen in `order_header`.
Ze verschijnen hier als één `location_id = NULL`-Unknown-bucket.

In [ ]:
@dp.materialized_view(
    name="daily_sales_by_location",
    comment=(
        "KPI aggregate: daily revenue per location from INTEGRATION.order_header. "
        "Grain: (order_date, location_id). NULL location_id rows kept as Unknown bucket. "
        "Materialised View — fully recomputed on each pipeline run."
    ),
)
def daily_sales_by_location():
    """Materialised View — dagelijkse omzet-KPI geaggregeerd per locatie.

    Leest INTEGRATION.order_header via spark.read (batch-semantiek).  Een
    @dp.materialized_view wordt volledig herberekend bij elke pipeline-run, zodat
    Silver-correcties automatisch propageren.

    Grain: één rij per (order_date, location_id)-combinatie.  NULL location_id-
    waarden worden opzettelijk bewaard — ze verschijnen als één Unknown-bucket die
    orders met onbekende locatie zichtbaar maakt voor analyst-triage.

    avg_order_value wordt gecast naar DECIMAL(38,4) na de deling om de kolomspec te
    matchen; de deling zelf promoveert naar Double, dus een expliciete cast is vereist.
    """
    oh = spark.read.table(f"{catalog}.{silver_schema}.order_header")

    agg = (
        oh
        .withColumn("order_date", F.col("order_ts").cast("date"))
        .groupBy("order_date", "location_id")
        .agg(
            F.count("*")                        .cast("bigint")         .alias("total_orders"),
            F.sum("order_total")                .cast("decimal(38,4)") .alias("total_revenue"),
            F.sum("order_tax_amount")           .cast("decimal(38,4)") .alias("total_tax"),
            F.sum("order_discount_amount")      .cast("decimal(38,4)") .alias("total_discount"),
        )
    )

    return agg.withColumn(
        "avg_order_value",
        (F.col("total_revenue") / F.col("total_orders")).cast("decimal(38,4)"),
    ).select(
        "order_date",
        "location_id",
        "total_orders",
        "total_revenue",
        "total_tax",
        "total_discount",
        "avg_order_value",
    )

## monthly_revenue_by_currency — Materialised View

Aggregeert `INTEGRATION.order_header` op `(year_month, order_currency)`-grain.

**Kolomspec:**

| Kolom | Type | Berekening |
|---|---|---|
| `year_month` | DATE | `DATE_TRUNC('month', order_ts)` — eerste dag van de maand |
| `order_currency` | STRING | groepeersleutel |
| `total_orders` | BIGINT | `COUNT(*)` |
| `total_revenue` | DECIMAL(38, 4) | `SUM(order_total)` |
| `avg_order_value` | DECIMAL(38, 4) | `total_revenue / total_orders` |

**year_month is een DATE, geen STRING:** `DATE_TRUNC('month', order_ts)` levert
de eerste dag van de maand op (bijv. `2024-03-01`).  Databricks-datumfuncties en
BI-tools (AI/BI Dashboard, Genie) werken veel beter met DATE dan met `'yyyy-MM'`-strings
voor tijdreeksassen en vergelijkingen.

In [ ]:
@dp.materialized_view(
    name="monthly_revenue_by_currency",
    comment=(
        "Monthly revenue trend per currency from INTEGRATION.order_header. "
        "Grain: (year_month, order_currency). year_month is DATE (first-of-month) "
        "— not a string. Materialised View — fully recomputed on each pipeline run."
    ),
)
def monthly_revenue_by_currency():
    """Materialised View — maandelijkse omzettrend geaggregeerd per valuta.

    Leest INTEGRATION.order_header via spark.read (batch-semantiek).  Een
    @dp.materialized_view wordt volledig herberekend bij elke pipeline-run, zodat
    Silver-correcties automatisch propageren.

    Grain: één rij per (year_month, order_currency)-combinatie.

    year_month wordt geproduceerd door DATE_TRUNC('month', order_ts), wat de
    eerste kalenderdag van de maand als DATE geeft (bijv. 2024-03-01).  Het gebruik
    van DATE in plaats van een 'yyyy-MM'-string laat BI-tools en Databricks-
    datumfuncties er direct op opereren zonder parsing.

    avg_order_value wordt gecast naar DECIMAL(38,4) na de deling om de kolomspec te
    matchen; de deling zelf promoveert naar Double, dus een expliciete cast is vereist.
    """
    oh = spark.read.table(f"{catalog}.{silver_schema}.order_header")

    agg = (
        oh
        .withColumn("year_month", F.date_trunc("month", F.col("order_ts")).cast("date"))
        .groupBy("year_month", "order_currency")
        .agg(
            F.count("*")           .cast("bigint")         .alias("total_orders"),
            F.sum("order_total")   .cast("decimal(38,4)") .alias("total_revenue"),
        )
    )

    return agg.withColumn(
        "avg_order_value",
        (F.col("total_revenue") / F.col("total_orders")).cast("decimal(38,4)"),
    ).select(
        "year_month",
        "order_currency",
        "total_orders",
        "total_revenue",
        "avg_order_value",
    )

## sales_lines_wide — Materialised View met Liquid Clustering

Brede, gedenormaliseerde, AI/BI-vriendelijke Materialised View op regelgrain.  Dit is de
Genie- en Dashboards-target.

**Bron:** `INTEGRATION.sales_line` — alle kolommen komen ongewijzigd door.  Zes afgeleide
kolommen worden toegevoegd voor tijdindeling en regel-niveau controle.

**Afgeleide kolomspec:**

| Afgeleide kolom | Type | Berekening |
|---|---|---|
| `order_date` | DATE | `CAST(order_ts AS DATE)` |
| `order_hour` | INT | `HOUR(order_ts)` (0–23) |
| `order_day_of_week` | STRING | `DATE_FORMAT(order_ts, 'EEEE')` (bijv. 'Monday') |
| `order_year_month` | DATE | eerste dag van de maand via `DATE_TRUNC('month', order_ts)` |
| `shift_duration_minutes` | INT | `(end_seconds - start_seconds) / 60` uit `'HH:mm:ss'`-strings |
| `line_subtotal` | DECIMAL(38, 4) | `quantity * unit_price` (controle ten opzichte van `price`) |

**Liquid Clustering:** `CLUSTER BY (truck_id, location_id, order_date, order_currency)`.
Dashboards en Genie filteren op verschillende combinaties van deze vier sleutels.
Liquid Clustering vermijdt de handmatige partitie-sleutelkeuze — Databricks herbalanceert
automatisch naarmate querypatronen evolueren.

**Patroon:** `@dp.materialized_view` met `spark.read` (batch, MV-semantiek).  De pipeline
herberekent volledig bij elke run.  Geen streaming-state nodig — regel-grain alleen-lezen
tabel.

In [ ]:
@dp.materialized_view(
    name="sales_lines_wide",
    comment=(
        "Wide, denormalised line-grain table for AI/BI Genie and Dashboards. "
        "All INTEGRATION.sales_line columns plus 6 derived columns: "
        "order_date, order_hour, order_day_of_week, order_year_month, "
        "shift_duration_minutes, line_subtotal. "
        "Liquid Clustering on (truck_id, location_id, order_date, order_currency). "
        "Materialised View — fully recomputed on each pipeline run."
    ),
    cluster_by=["truck_id", "location_id", "order_date", "order_currency"],
)
def sales_lines_wide():
    """Materialised View — brede, gedenormaliseerde regelgrain-tabel voor AI/BI-consumption.

    Leest INTEGRATION.sales_line via spark.read (batch-semantiek).  Alle bronkolommen
    komen ongewijzigd door; zes afgeleide kolommen worden toegevoegd via de
    pure hulpfuncties uit derivation_helpers.

    Afgeleide kolommen:
    - order_date          : DATE   — CAST(order_ts AS DATE)
    - order_hour          : INT    — HOUR(order_ts) 0–23
    - order_day_of_week   : STRING — DATE_FORMAT(order_ts, 'EEEE') bijv. 'Monday'
    - order_year_month    : DATE   — DATE_TRUNC('month', order_ts) eerste-van-de-maand
    - shift_duration_minutes : INT — (end_seconds - start_seconds) / 60 uit 'HH:mm:ss'
    - line_subtotal       : DECIMAL(38,4) — quantity * unit_price

    Liquid Clustering is gedeclareerd via cluster_by= zodat de tabeleigenschap
    clusteringColumns automatisch door de pipeline wordt ingesteld — geen handmatige
    ALTER TABLE nodig.  Clustersleutels gekozen op basis van dashboard/Genie-
    filterpatronen: truck_id, location_id, order_date, order_currency.
    """
    sl = spark.read.table(f"{catalog}.{silver_schema}.sales_line")

    return (
        sl
        # ------------------------------------------------------------------
        # Afgeleide tijdkolommen (derivation_helpers)
        # ------------------------------------------------------------------
        .withColumn("order_date",            derive_order_date(F.col("order_ts")))
        .withColumn("order_hour",            derive_order_hour(F.col("order_ts")))
        .withColumn("order_day_of_week",     derive_order_day_of_week(F.col("order_ts")))
        .withColumn("order_year_month",      derive_order_year_month(F.col("order_ts")))
        # ------------------------------------------------------------------
        # Shiftduur — geparseerd uit Silver 'HH:mm:ss'-strings
        # ------------------------------------------------------------------
        .withColumn(
            "shift_duration_minutes",
            derive_shift_duration_minutes(F.col("shift_start_time"), F.col("shift_end_time")),
        )
        # ------------------------------------------------------------------
        # Regelsubtotaal — controle ten opzichte van Bronze price-kolom
        # ------------------------------------------------------------------
        .withColumn(
            "line_subtotal",
            derive_line_subtotal(F.col("quantity"), F.col("unit_price")),
        )
        # ------------------------------------------------------------------
        # Expliciete kolomvolgorde: bronkolommen eerst, dan afgeleide kolommen
        # ------------------------------------------------------------------
        .select(
            # Regelgrain-sleutel
            "order_detail_id",
            "order_id",
            # order_detail-bedrijfskolommen
            "menu_item_id",
            "quantity",
            "unit_price",
            "price",
            "order_item_discount_amount",
            "line_number",
            # order_header-bedrijfskolommen (gedenormaliseerd)
            "truck_id",
            "location_id",
            "customer_id",
            "discount_id",
            "shift_id",
            "shift_start_time",
            "shift_end_time",
            "order_channel",
            "order_ts",
            "served_ts",
            "order_currency",
            "order_amount",
            "order_tax_amount",
            "order_discount_amount",
            "order_total",
            # 6 afgeleide kolommen
            "order_date",
            "order_hour",
            "order_day_of_week",
            "order_year_month",
            "shift_duration_minutes",
            "line_subtotal",
        )
    )

## Demo-notities

- **DLT-grafiekweergave**: Deze pipeline is geregistreerd als een aparte DLT-pipeline
  (`resources/dlt_datamart.yml`, doelschema `DATAMART`).  De Databricks-UI toont vier
  MV-nodes: drie aggregaten die lezen vanuit `INTEGRATION.order_header`
  (`daily_sales_by_truck`, `daily_sales_by_location`, `monthly_revenue_by_currency`)
  en één brede regelgrain-MV die leest vanuit `INTEGRATION.sales_line` (`sales_lines_wide`).
- **NULL truck_id / location_id demo**: Voeg een Bronze-rij in met `TRUCK_ID = NULL`
  of `LOCATION_ID = NULL` (of laat ze weg).  Na de end-to-end workflow-run bevat het
  bijbehorende aggregaat een rij waarbij de sleutel IS NULL met het juiste aantal orders
  en de juiste omzet — de opzettelijke Unknown-bucket.
- **MV-propagatie demo**: Corrigeer een `order_total` in `STAGING_AZURESTORAGE` en
  herstart de workflow.  Silver propageert de correctie via CDF + create_auto_cdc_flow;
  de volgende Gold-pipeline-run herberekent alle vier MV's volledig, waardoor de
  bijgewerkte aggregaten en de gecorrigeerde brede-tabelrij worden opgepikt.
- **year_month als DATE demo**: Bevraag `monthly_revenue_by_currency` en gebruik een
  datumvergelijking (`WHERE year_month >= '2024-01-01'`) — geen string-parsing vereist.
  BI-tools renderen de DATE-kolom automatisch op een tijdas.
- **Liquid Clustering demo**: Voer `SHOW TBLPROPERTIES DEMO.DATAMART.sales_lines_wide` uit
  en bevestig dat `clusteringColumns` de vier clustersleutels vermeldt
  (`truck_id, location_id, order_date, order_currency`).  Geen handmatige `ALTER TABLE`
  nodig — `cluster_by=` op de `@dp.materialized_view`-declaratie stelt de eigenschap in.
- **Workflow-keten**: `setup → ingest_* → dlt_integration → dlt_datamart`.
  De `dlt_datamart`-taak is afhankelijk van `dlt_integration` zodat Silver altijd
  up-to-date is vóórdat de Gold-aggregatie wordt uitgevoerd.